# Plug in a driver, step by step

A walk-through for someone who would like to connect a microscope to the
ZMART Controller. It plugs in the mock driver first, looks at what a driver
is made of, and then builds a small driver of your own in a single file. The
[README](README.md) of this part is the complete reference: every function a
driver must have, what each must answer, and the rules behind them.

**Before you start**

- Python 3.11 or newer, with the controller installed:
  `pip install "git+https://github.com/thomdehoog/ZMART-controller"`
- No microscope. Everything here runs on a laptop.

Run the cells one at a time, from top to bottom.

## The idea

A workflow written for one microscope does not run on another, because every
microscope software speaks its own language. The controller stands in
between. Your workflow only ever speaks the controller's short list of
commands, and a **driver** translates each command for one particular
microscope.

```
your workflow ──► zmart_controller ──► driver ──► vendor software ──► microscope
```

Towards the controller, a driver offers **one Python function per command**:
`connect`, `get_xyz`, `set_xyz`, `acquire`, and so on. The controller calls
those functions. Plugging in a driver means telling the controller where
they are.

## Step 1: plug in the mock driver

The controller comes with one driver: the mock, a simulated microscope. Plug
it in and look around.

In [1]:
import zmart_controller

zmart_controller.set_instrument("mock", {"output_root": "images"})
zmart_controller.get_actuators()["content"]

{'x': ['motoric'], 'y': ['motoric'], 'z': ['motoric', 'piezo']}

These are the motors that can move each axis. z has two: a motor for long
moves and a piezo for fine, fast steps.

Now move the stage and take a picture. Every command answers in the same
shape, `{"success": ..., "content": ...}`.

In [2]:
zmart_controller.set_xyz(100, 50, 0)
answer = zmart_controller.acquire(position_label="A1")
answer["content"]["files"]

['images/A1.ome.tif', 'images/A1.commands.json']

In [3]:
zmart_controller.disconnect()

Part 2, [Use the controller](../2_use_the_controller/tutorial.ipynb), goes
through all the commands. Here we look at the other side: the driver.

## Step 2: look at what a driver is

The controller needs these functions from every driver. It finds them by
name.

In [4]:
zmart_controller.utils.OPS

('connect',
 'get_acquisition_settings',
 'get_actuators',
 'get_xyz',
 'set_xyz',
 'acquire',
 'get_state',
 'set_state',
 'get_procedures',
 'run_procedure',
 'get_info')

These eleven names, plus an optional `disconnect`, are the whole connection
between the controller and a microscope.

In the mock driver, they live in the file `zmart_controller_plugin.py` of
the package `zmart_controller.mock`. Each one is short. It takes the request,
hands it to the parts of the driver below it (moving the stage, reading back
the position, writing the file), and wraps the result as
`{"success": ..., "content": ...}`. The other folders of the mock are those
parts. A real microscope needs them, because its software is slow, sometimes
busy, and sometimes wrong. A pretend microscope of your own does not, which
is what the next step shows.

## Step 3: write a driver of your own

The cell below writes a file called `pretend_scope.py` next to this notebook
(that is what the first line, `%%writefile`, does). It is a complete driver
for a microscope that only exists in memory. It remembers where its stage is
and what its exposure is. Each "picture" is a small text file that says where
and how it was taken.

In [5]:
%%writefile pretend_scope.py
"""A ZMART driver for a pretend microscope, in one file.

The microscope only exists in memory. It remembers where its stage is and
what its exposure is. Each "picture" is a small text file that says where
and how it was taken.
"""

import json
from pathlib import Path

# The driver's name in the controller's list of drivers.
NAME = "pretend-scope"

# How far each axis can travel, in micrometres from the origin.
TRAVEL = {"x": [-1000.0, 1000.0], "y": [-1000.0, 1000.0], "z": [-100.0, 100.0]}

# Half a field of view in x and y. A picture taken at the edge of the travel
# still shows this much further out. This microscope takes no z-stacks, so 0 in z.
HALF_FIELD = {"x": 50.0, "y": 50.0, "z": 0.0}


def _answer(content, success=True):
    """Wrap the content in the shape every command returns."""
    return {"success": success, "content": content}


def connect(connection):
    """Start the pretend microscope. The handle is a plain dict that holds its state."""
    output_root = Path(connection.get("output_root", "pretend_scope_images"))
    output_root.mkdir(parents=True, exist_ok=True)
    return {
        "position": {"x": 0.0, "y": 0.0, "z": 0.0},
        "settings": {"exposure_ms": 10.0},
        "output_root": output_root,
    }


def get_info(handle):
    return _answer(
        {
            "output_root": str(handle["output_root"]),
            "description": "A pretend microscope that lives in memory. One motor per axis. "
            "exposure_ms is the exposure time in milliseconds. +z points towards the objective.",
        }
    )


def get_actuators(handle):
    return _answer({axis: ["motor"] for axis in TRAVEL})


def get_xyz(handle, *, with_actuators=None):
    reading = {}
    for axis, (low, high) in TRAVEL.items():
        reading[axis] = {
            "value": handle["position"][axis],
            "actuator": "motor",
            "canvas": [low - HALF_FIELD[axis], high + HALF_FIELD[axis]],
        }
    return _answer(reading)


def set_xyz(handle, x, y, z, *, with_actuators=None):
    target = {"x": x, "y": y, "z": z}
    for axis, value in target.items():
        low, high = TRAVEL[axis]
        if not low <= value <= high:
            raise ValueError(f"{axis} = {value} um is outside the travel, {low} to {high} um")
    handle["position"] = target
    return _answer({"position": dict(target), "actuators": {axis: "motor" for axis in target}})


def get_state(handle):
    return _answer({"changeable": dict(handle["settings"]), "observed": {"objective": "10x"}})


def set_state(handle, state):
    applied = {}
    for name, value in state.get("changeable", {}).items():
        if name not in handle["settings"]:
            raise ValueError(f"unknown setting {name!r}")
        handle["settings"][name] = value
        applied[name] = value
    return _answer({"applied": applied})


def get_acquisition_settings(handle):
    return _answer({"format": {"options": ["json"], "active": "json"}})


def acquire(handle, *, position_label, acquisition_settings=None):
    for name, value in (acquisition_settings or {}).items():
        if name != "format" or value != "json":
            raise ValueError(f"unknown acquisition setting {name!r} = {value!r}")
    path = handle["output_root"] / f"{position_label}.json"
    path.write_text(json.dumps({"position": handle["position"], "settings": handle["settings"]}))
    where = handle["position"]
    plane = {
        "path": str(path),
        "c": 0,
        "z": 0,
        "t": 0,
        "x_um": where["x"],
        "y_um": where["y"],
        "z_um": where["z"],
    }
    return _answer({"position_label": position_label, "files": [str(path)], "planes": [plane]})


def get_procedures(handle):
    return _answer({})


def run_procedure(handle, procedure):
    raise ValueError(f"unknown procedure {procedure.get('name')!r}; this microscope has none")

Writing pretend_scope.py


A few things to notice, because a real driver does the same:

- **`connect` returns a handle**, here a plain dictionary. The controller
  never looks inside it. It only hands it back to every other function. A
  real driver keeps its connection to the vendor software in it.
- **`set_xyz` refuses a position outside the travel by raising
  `ValueError`.** It never moves "as far as it can". A workflow that asked for
  somewhere else must hear about it.
- **`acquire` says where it saved the picture** (`files`) and where on the
  sample it was taken (`planes`). That is how a workflow finds its pictures on
  any microscope.
- **`canvas` is the travel widened by half a field**, because a picture
  taken at the edge of the travel still shows a little beyond it.

## Step 4: plug it in

Plug in your driver exactly as you plugged in the mock. While you write a
driver, you hand the module itself to `set_instrument`.

In [6]:
import pretend_scope

zmart_controller.set_instrument(pretend_scope)

zmart_controller.set_xyz(100, 50, 0)
answer = zmart_controller.acquire(position_label="A1")
answer["content"]["files"]

['pretend_scope_images/A1.json']

The "picture" records the position and the exposure.

In [7]:
print(open(answer["content"]["files"][0]).read())

{"position": {"x": 100, "y": 50, "z": 0}, "settings": {"exposure_ms": 10.0}}


Now ask for a position outside the travel. Your driver refuses, and the
controller passes the refusal on unchanged.

In [8]:
try:
    zmart_controller.set_xyz(5000, 0, 0)
except ValueError as error:
    print(error)

x = 5000 um is outside the travel, -1000.0 to 1000.0 um


## Step 5: check that it fits

The controller can check a driver's answers against the requirements.
`validate_driver` connects, calls every `get_*` function, and lists any
problem in plain words. An empty list means every answer has the right shape.

In [9]:
zmart_controller.validate_driver(pretend_scope)

[]

It moves nothing and acquires nothing, so check one acquisition separately.

In [10]:
answer = zmart_controller.acquire(position_label="A2")
zmart_controller.check_acquire_answer(answer)

[]

To see what a problem looks like, hand in a driver with a mistake. A driver
can also be a plain dictionary of functions, so copy ours and swap in a
`get_xyz` that forgets the canvas.

In [11]:
functions = {name: getattr(pretend_scope, name) for name in zmart_controller.utils.OPS}


def get_xyz_without_canvas(handle, *, with_actuators=None):
    answer = pretend_scope.get_xyz(handle, with_actuators=with_actuators)
    for axis in answer["content"].values():
        del axis["canvas"]
    return answer


functions["get_xyz"] = get_xyz_without_canvas
zmart_controller.validate_driver(functions)

["get_xyz: axis 'x' is missing 'canvas'",
 "get_xyz: axis 'y' is missing 'canvas'",
 "get_xyz: axis 'z' is missing 'canvas'"]

This is the loop to work in when you write a driver: change a function,
validate, read the problems, repeat.

## Step 6: register it

So far you handed the module to `set_instrument` yourself. To use the driver
like any other, register it once. In a real driver the file is called
`zmart_controller_plugin.py`. Any name works for registering.

In [12]:
zmart_controller.register_driver("pretend_scope.py")
zmart_controller.get_drivers()

['mock', 'pretend-scope']

From now on, every Python session on this computer can plug it in by name.

In [13]:
zmart_controller.set_instrument("pretend-scope")
zmart_controller.get_info()["content"]["description"]

'A pretend microscope that lives in memory. One motor per axis. exposure_ms is the exposure time in milliseconds. +z points towards the objective.'

In [14]:
zmart_controller.disconnect()

This tutorial's driver does not need to stay on the list. Take it off again.

In [15]:
zmart_controller.remove_driver("pretend-scope")
zmart_controller.get_drivers()

['mock']

## Step 7: from a pretend microscope to a real one

Your file already works with every workflow written for the controller. For
a real microscope, each function does the same job, but by talking to the
vendor software instead of a dictionary:

- **`connect`** starts or logs into the vendor software, and loads what was
  measured once on this microscope: the origin, the travel limits, the
  calibration. These live on the microscope computer, in the folder that
  `zmart_controller.utils.config_root()` names.
- **`set_xyz`** checks the limits, sends the move, and reads the position
  back until it matches, because microscope software often says "done" before
  the stage has stopped.
- **`acquire`** starts the capture, waits until the vendor's file is
  complete, turns it into OME-TIFF or OME-Zarr lined up with the stage, and
  reports `files` and `planes`.

Once a driver grows beyond one file, make it a package: a folder whose
`__init__.py` imports the functions, exactly as the mock does.
[The anatomy of a ZMART driver](https://github.com/thomdehoog/ZMART-drivers/blob/main/docs/driver-anatomy.md)
describes how our own drivers are built inside. The mock driver is a complete
example of it.

## When something goes wrong

| Message | Meaning | What to do |
|---|---|---|
| `ValueError: driver ... is missing functions: [...]` | A required function is missing or misspelled | Match the names in the README exactly |
| `ModuleNotFoundError: No module named 'pretend_scope'` | Python cannot see the file | Start the notebook in the folder that holds `pretend_scope.py` |
| `validate_driver` lists problems | Each sentence names the function and what is missing | Fix that function and validate again |
| An error from your own code | The controller passes everything your driver raises on unchanged | The message points at your code |

## Good to know

- **The names are the contract.** The controller finds the functions by
  name. A helper function of your own must not use one of the twelve names.
- **Raise when carrying on is unsafe.** `ValueError` for a request that is
  wrong, `RuntimeError` for a microscope that fails. Use `success: False` only
  for an outcome a workflow can safely carry on from.
- **Positions are micrometres from the origin.** In a saved picture, right is
  +x and down is +y, on every microscope. A real driver turns the vendor's own
  numbers and image orientation into this frame.
- **Never overwrite a picture.** The pretend driver above does, to stay
  short. A real one, like the mock, saves a second acquisition with the same
  label as `A1_001` instead.

## Where to go next

- The [README](README.md): the full requirements for every function.
- Part 2, [Use the controller](../2_use_the_controller/tutorial.ipynb):
  every command, on the mock.

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.